# 02 — Data Cleaning Pipeline
**Project:** FIFA World Cup Analytics  
**Notebook:** `02_cleaning.ipynb`  

## What this notebook does
1. Loads all three raw datasets from `data/raw/`
2. Merges them into a single **player-match-tournament** dataset using logical join keys
3. Applies a documented, step-by-step cleaning pipeline
4. Exports the clean dataset to the configured output directory.

## Merge Logic
```
WorldCupPlayers
    └── LEFT JOIN WorldCupMatches  ON  RoundID + MatchID   → adds match context per player
            └── LEFT JOIN WorldCups  ON  Year               → adds tournament context per match
```
**Grain of the final dataset:** One row = One player in one match in one tournament.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
## 1. Setup & Imports

In [3]:
import pandas as pd
import numpy as np
import os

os.makedirs('../data/processed', exist_ok=True)
pd.set_option('display.max_columns', None)

print('Setup complete. Output will be saved to: ../data/processed/')

Setup complete. Output will be saved to: ../data/processed/


---
## 2. Load Raw Data

Raw files are read-only. All cleaning is done on copies.

In [4]:
matches_raw   = pd.read_csv('../data/raw/WorldCupMatches.csv')
players_raw   = pd.read_csv('../data/raw/WorldCupPlayers.csv')
worldcups_raw = pd.read_csv('../data/raw/WorldCups.csv')

print('Raw shapes loaded:')
print(f'  WorldCupMatches  : {matches_raw.shape}')
print(f'  WorldCupPlayers  : {players_raw.shape}')
print(f'  WorldCups        : {worldcups_raw.shape}')

Raw shapes loaded:
  WorldCupMatches  : (4572, 20)
  WorldCupPlayers  : (37784, 9)
  WorldCups        : (20, 10)


---
## 3. Pre-Merge Preparation

Before merging, fix the join keys in WorldCupMatches:
- Drop 3,720 fully blank rows (NaN in every column — zero data value)
- Drop duplicate rows
- Cast `Year`, `RoundID`, `MatchID` from float to int so they match WorldCupPlayers exactly

In [5]:
# --- WorldCupMatches: minimal prep for join ---
matches = matches_raw.copy()

# Drop fully blank rows
before = len(matches)
matches.dropna(how='all', inplace=True)
print(f'Blank rows removed from Matches : {before - len(matches)}')

# Drop duplicates
before = len(matches)
matches.drop_duplicates(inplace=True)
print(f'Duplicate rows removed from Matches : {before - len(matches)}')

# Cast join keys to int (were float because of NaN rows)
matches['Year']    = matches['Year'].astype(int)
matches['RoundID'] = matches['RoundID'].astype(int)
matches['MatchID'] = matches['MatchID'].astype(int)
print(f'Matches ready for join: {matches.shape}')

# --- WorldCups: rename columns that would clash after merge ---
# 'Attendance' exists in both Matches and WorldCups
worldcups = worldcups_raw.copy()
worldcups.rename(columns={
    'Attendance' : 'tournament_attendance',
    'Country'    : 'host_country'
}, inplace=True)
print(f'WorldCups ready for join: {worldcups.shape}')

Blank rows removed from Matches : 3720
Duplicate rows removed from Matches : 16
Matches ready for join: (836, 20)
WorldCups ready for join: (20, 10)


---
## 4. Merge — Build the Master Dataset

**Join 1:** WorldCupPlayers + WorldCupMatches on `[RoundID, MatchID]`
> Every player record gets full match context: stadium, stage, goals, referee, attendance.

**Join 2:** Result + WorldCups on `Year`
> Every player-match record gets tournament context: host country, winner, qualified teams.

In [6]:
players = players_raw.copy()

# JOIN 1 — Players + Matches
df = players.merge(matches, on=['RoundID', 'MatchID'], how='left')
print(f'After Join 1 (Players + Matches) : {df.shape}')

# JOIN 2 — Result + WorldCups
df = df.merge(worldcups, on='Year', how='left')
print(f'After Join 2 (+ WorldCups)       : {df.shape}')

print(f'\nFinal columns ({len(df.columns)}): {list(df.columns)}')

After Join 1 (Players + Matches) : (37784, 27)
After Join 2 (+ WorldCups)       : (37784, 36)

Final columns (36): ['RoundID', 'MatchID', 'Team Initials', 'Coach Name', 'Line-up', 'Shirt Number', 'Player Name', 'Position', 'Event', 'Year', 'Datetime', 'Stage', 'Stadium', 'City', 'Home Team Name', 'Home Team Goals', 'Away Team Goals', 'Away Team Name', 'Win conditions', 'Attendance', 'Half-time Home Goals', 'Half-time Away Goals', 'Referee', 'Assistant 1', 'Assistant 2', 'Home Team Initials', 'Away Team Initials', 'host_country', 'Winner', 'Runners-Up', 'Third', 'Fourth', 'GoalsScored', 'QualifiedTeams', 'MatchesPlayed', 'tournament_attendance']


---
## 5. Pre-Cleaning Audit on Merged Dataset

In [7]:
print('===== MERGED DATASET — PRE-CLEANING AUDIT =====')
print(f'Shape      : {df.shape}')
print(f'Duplicates : {df.duplicated().sum()}')
print()
print('Missing values per column:')
missing = df.isnull().sum()
print(missing[missing > 0].to_string())
print()
print('Data types:')
print(df.dtypes.to_string())

===== MERGED DATASET — PRE-CLEANING AUDIT =====
Shape      : (37784, 36)
Duplicates : 736

Missing values per column:
Position      33641
Event         28715
Attendance       92

Data types:
RoundID                    int64
MatchID                    int64
Team Initials             object
Coach Name                object
Line-up                   object
Shirt Number               int64
Player Name               object
Position                  object
Event                     object
Year                       int64
Datetime                  object
Stage                     object
Stadium                   object
City                      object
Home Team Name            object
Home Team Goals          float64
Away Team Goals          float64
Away Team Name            object
Win conditions            object
Attendance               float64
Half-time Home Goals     float64
Half-time Away Goals     float64
Referee                   object
Assistant 1               object
Assistant 2      

---
## 6. Cleaning Pipeline

Every step is logged with what was done and why.

| Step | Column(s) | Issue | Fix |
|------|-----------|-------|-----|
| 1 | All rows | 736 duplicate rows | Drop duplicates |
| 2 | All columns | Spaces in names | Rename to snake_case |
| 3 | assistant_1, assistant_2 | Not analytically useful | Drop columns |
| 4 | datetime | Plain string | Parse to datetime64 |
| 5 | goal cols, half-time cols | float (from blank rows) | Cast to int |
| 6 | attendance | 92 missing + float | Year-median fill, cast to int |
| 7 | win_conditions | Empty string | Fill with 'Normal' |
| 8 | shirt_number | 0 as placeholder | Replace 0 with 'Unknown' |
| 9 | position | 33,641 NaN | Fill with 'Unknown' |
| 10 | event | 28,715 NaN | Fill with 'No Event' |
| 11 | line_up | Opaque codes S/N | Decode to Starting/Substitute |
| 12 | tournament_attendance | String '590.549' | Remove separator dot, cast to int |
| 13 | All string cols | Whitespace | Strip all string columns |
| 14 | — | Missing KPI columns | Add total_goals, match_result, is_goal_scorer |

In [8]:
# STEP 1 — Remove duplicate rows
# Reason: 736 exact duplicate rows exist from the raw WorldCupPlayers export.
# Keeping them inflates player counts, event counts, and all aggregations.

before = len(df)
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'[STEP 1] Removed {before - len(df)} duplicate rows. Rows now: {len(df)}')

[STEP 1] Removed 736 duplicate rows. Rows now: 37048


In [9]:
# STEP 2 — Standardise all column names to snake_case
# Reason: Column names like 'Home Team Name' require bracket notation and
# cause confusion in Python, Pandas, and Tableau. snake_case is the standard.

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('-', '_', regex=False)
)

print(f'[STEP 2] All {len(df.columns)} columns renamed to snake_case:')
print(list(df.columns))

[STEP 2] All 36 columns renamed to snake_case:
['roundid', 'matchid', 'team_initials', 'coach_name', 'line_up', 'shirt_number', 'player_name', 'position', 'event', 'year', 'datetime', 'stage', 'stadium', 'city', 'home_team_name', 'home_team_goals', 'away_team_goals', 'away_team_name', 'win_conditions', 'attendance', 'half_time_home_goals', 'half_time_away_goals', 'referee', 'assistant_1', 'assistant_2', 'home_team_initials', 'away_team_initials', 'host_country', 'winner', 'runners_up', 'third', 'fourth', 'goalsscored', 'qualifiedteams', 'matchesplayed', 'tournament_attendance']


In [10]:
# STEP 3 — Drop analytically irrelevant columns
# Reason: 'assistant_1' and 'assistant_2' are linesman names.
# They have no relevance to player performance, match outcomes, or tournament trends.

df.drop(columns=['assistant_1', 'assistant_2'], inplace=True)

print(f'[STEP 3] Dropped assistant_1 and assistant_2. Columns remaining: {len(df.columns)}')

[STEP 3] Dropped assistant_1 and assistant_2. Columns remaining: 34


In [11]:
# STEP 4 — Parse 'datetime' string to proper datetime type AND drop nulls
# Reason: Stored as plain string '13 Jul 1930 - 15:00'.
# Parsing it enables date sorting, extracting match month, and time-series analysis in EDA.
# Rows with null datetimes are dropped as they are invalid for time-based analysis.

df['datetime'] = pd.to_datetime(
    df['datetime'].astype(str).str.strip(),
    format='%d %b %Y - %H:%M',
    errors='coerce'
)

# Count nulls before dropping to log them
nat_count = df['datetime'].isna().sum()

# Drop rows where datetime is null (NaT) and reset the index
df.dropna(subset=['datetime'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'[STEP 4] datetime parsed. Dropped {nat_count} rows with null (NaT) datetime.')

[STEP 4] datetime parsed. Dropped 454 rows with null (NaT) datetime.


In [12]:
# STEP 5 — Cast goal columns from float to int
# Reason: Goals are whole numbers. They were stored as float64 because the blank
# rows in the raw Matches file introduced NaN, which forced pandas to use float.
# Now that blank rows are removed, integer is the correct type.

goal_cols = [
    'home_team_goals', 'away_team_goals',
    'half_time_home_goals', 'half_time_away_goals'
]
for col in goal_cols:
    df[col] = df[col].astype(int)

print(f'[STEP 5] Cast to int: {goal_cols}')

[STEP 5] Cast to int: ['home_team_goals', 'away_team_goals', 'half_time_home_goals', 'half_time_away_goals']


In [13]:
# STEP 6 — Fill missing match Attendance and cast to int
# Reason: 92 attendance values are null after the merge.
# Dropping those rows would remove valid player records.
# Year-level median fill is used because attendance is right-skewed
# (large finals distort the mean; median is more representative).

print(f'[STEP 6] Attendance nulls before fill: {df["attendance"].isna().sum()}')

df['attendance'] = df.groupby('year')['attendance'].transform(
    lambda x: x.fillna(x.median())
)
df['attendance'] = df['attendance'].astype(int)

print(f'         Attendance nulls after fill : {df["attendance"].isna().sum()}')

[STEP 6] Attendance nulls before fill: 46
         Attendance nulls after fill : 0


In [14]:
# STEP 7 — Fill empty Win Conditions with 'Normal'
# Reason: win_conditions is only filled for Extra Time or Penalty matches.
# Empty = regular-time decision. Filling with 'Normal' makes groupby and
# Tableau filters work correctly without null-related edge cases.

df['win_conditions'] = (
    df['win_conditions']
    .astype(str).str.strip()
    .replace('', 'Normal')
    .replace('nan', 'Normal')
)

print(f'[STEP 7] win_conditions distribution:')
print(df['win_conditions'].value_counts().head(8).to_string())

[STEP 7] win_conditions distribution:
win_conditions
Normal                             34087
Italy win after extra time           173
Win on Golden Goal                   138
Argentina win after extra time       136
England win after extra time         132
Germany win after extra time          92
Belgium win after extra time          90
Brazil win on penalties (3 - 2)       90


In [15]:
# STEP 8 — Replace Shirt Number = 0 with 'Unknown'
# Reason: Shirt number 0 is not a valid football jersey number.
# It was used as a placeholder for unknown/unrecorded values.
# Replacing it with 'Unknown' preserves the record without introducing nulls.

zero_shirts = (df['shirt_number'] == 0).sum()
df['shirt_number'] = df['shirt_number'].replace(0, 'Unknown')

print(f'[STEP 8] Replaced {zero_shirts} shirt_number = 0 entries with "Unknown".')

[STEP 8] Replaced 3069 shirt_number = 0 entries with "Unknown".


In [16]:
# STEP 9 — Fill missing Position with 'Unknown'
# Reason: Position (GK/DF/MF/FW) was not consistently recorded before the 1980s.
# 33,641 nulls affect the majority of records. Dropping them guts the dataset.
# 'Unknown' preserves every row for match-level, event-level, and temporal analysis.

missing_pos = df['position'].isna().sum()
df['position'] = df['position'].fillna('Unknown')

print(f'[STEP 9] Filled {missing_pos} nulls with "Unknown".')
print(df['position'].value_counts().to_string())

[STEP 9] Filled 32626 nulls with "Unknown".
position
Unknown    32626
GK          2318
C           1463
GKC          187


In [17]:
# STEP 10 — Fill missing Event with 'No Event'
# Reason: event records goals (G40'), yellow cards (Y65'), and substitutions (SU78').
# A null means the player had no recorded action — fully expected for most players.
# 'No Event' makes the column filterable and groupable without null handling.

missing_ev = df['event'].isna().sum()
df['event'] = df['event'].fillna('No Event')

print(f'[STEP 10] Filled {missing_ev} nulls with "No Event".')

[STEP 10] Filled 27925 nulls with "No Event".


In [18]:
# STEP 11 — Decode Line-up codes to readable labels
# Reason: 'S' and 'N' are opaque. Replacing with 'Starting' and 'Substitute'
# makes the column self-documenting in dashboards and the project report.

df['line_up'] = df['line_up'].map({'S': 'Starting', 'N': 'Substitute'}).fillna(df['line_up'])

print(f'[STEP 11] line_up after decode:')
print(df['line_up'].value_counts().to_string())

[STEP 11] line_up after decode:
line_up
Substitute    18422
Starting      18172


In [19]:
# STEP 12 — Convert tournament_attendance from string to integer
# Reason: WorldCups stores this as '590.549' — European-style where dot = thousands separator.
# This means 590,549 total spectators. Remove the dot and cast to int.

print(f'[STEP 12] Before: {df["tournament_attendance"].unique()[:4]}')

df['tournament_attendance'] = (
    df['tournament_attendance']
    .astype(str)
    .str.replace('.', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(int)
)

print(f'         After : {df["tournament_attendance"].unique()[:4]}')

[STEP 12] Before: ['590.549' '363.000' '375.700' '1.045.246']
         After : [ 590549  363000  375700 1045246]


In [20]:
# STEP 13 — Strip whitespace from all string columns
# Reason: Team names, city names, and player names contain trailing/leading spaces
# e.g. 'Brazil ' != 'Brazil'. These cause silent mismatches in groupby and Tableau filters.

str_cols = df.select_dtypes(include='object').columns
for col in str_cols:
    df[col] = df[col].astype(str).str.strip()

print(f'[STEP 13] Whitespace stripped from {len(str_cols)} columns.')

[STEP 13] Whitespace stripped from 21 columns.


In [21]:
# STEP 14 — Add derived analytical columns
# Reason: These are direct KPI inputs computable from the clean data.
# They reduce repeated logic in EDA/statistical notebooks and power Tableau calculated fields.

# 14a — Total goals per match
df['total_goals'] = df['home_team_goals'] + df['away_team_goals']

# 14b — Match result from each player's team perspective
def get_result(row):
    if row['team_initials'] == row['home_team_initials']:
        mine, theirs = row['home_team_goals'], row['away_team_goals']
    else:
        mine, theirs = row['away_team_goals'], row['home_team_goals']
    if mine > theirs: return 'Win'
    if mine < theirs: return 'Loss'
    return 'Draw'

df['match_result'] = df.apply(get_result, axis=1)

# 14c — Goal scorer flag
df['is_goal_scorer'] = df['event'].str.contains(r'G\d', regex=True).astype(int)

print('[STEP 14] Derived columns added:')
print(f'  total_goals    : {df["total_goals"].min()} to {df["total_goals"].max()}')
print(f'  match_result   : {df["match_result"].value_counts().to_dict()}')
print(f'  is_goal_scorer : {df["is_goal_scorer"].value_counts().to_dict()}')

[STEP 14] Derived columns added:
  total_goals    : 0 to 12
  match_result   : {'Win': 14237, 'Loss': 14162, 'Draw': 8195}
  is_goal_scorer : {0: 34768, 1: 1826}


---
## 7. Post-Cleaning Audit

In [22]:
print('===== MASTER DATASET — POST CLEANING =====')
print(f'Shape      : {df.shape}')
print(f'Duplicates : {df.duplicated().sum()}')
print()
remaining_nulls = df.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
if len(remaining_nulls):
    print('Remaining nulls (shirt_number only is expected):')
    print(remaining_nulls.to_string())
else:
    print('Missing values: None')
print()
print('Dtypes:')
print(df.dtypes.to_string())

===== MASTER DATASET — POST CLEANING =====
Shape      : (36594, 37)
Duplicates : 0

Missing values: None

Dtypes:
roundid                           int64
matchid                           int64
team_initials                    object
coach_name                       object
line_up                          object
shirt_number                     object
player_name                      object
position                         object
event                            object
year                              int64
datetime                 datetime64[ns]
stage                            object
stadium                          object
city                             object
home_team_name                   object
home_team_goals                   int64
away_team_goals                   int64
away_team_name                   object
win_conditions                   object
attendance                        int64
half_time_home_goals              int64
half_time_away_goals              int64
refere

In [23]:
df.head(5)

,roundid,matchid,team_initials,coach_name,line_up,shirt_number,player_name,position,event,year,datetime,stage,stadium,city,home_team_name,home_team_goals,away_team_goals,away_team_name,win_conditions,attendance,half_time_home_goals,half_time_away_goals,referee,home_team_initials,away_team_initials,host_country,winner,runners_up,third,fourth,goalsscored,qualifiedteams,matchesplayed,tournament_attendance,total_goals,match_result,is_goal_scorer
0,201,1096,FRA,CAUDRON Raoul (FRA),Starting,Unknown,Alex THEPOT,GK,No Event,1930,1930-07-13 15:00:00,Group 1,Pocitos,Montevideo,France,4,1,Mexico,Normal,4444,3,0,LOMBARDI Domingo (URU),FRA,MEX,Uruguay,Uruguay,Argentina,USA,Yugoslavia,70,13,18,590549,5,Win,0
1,201,1096,MEX,LUQUE Juan (MEX),Starting,Unknown,Oscar BONFIGLIO,GK,No Event,1930,1930-07-13 15:00:00,Group 1,Pocitos,Montevideo,France,4,1,Mexico,Normal,4444,3,0,LOMBARDI Domingo (URU),FRA,MEX,Uruguay,Uruguay,Argentina,USA,Yugoslavia,70,13,18,590549,5,Loss,0
2,201,1096,FRA,CAUDRON Raoul (FRA),Starting,Unknown,Marcel LANGILLER,Unknown,G40',1930,1930-07-13 15:00:00,Group 1,Pocitos,Montevideo,France,4,1,Mexico,Normal,4444,3,0,LOMBARDI Domingo (URU),FRA,MEX,Uruguay,Uruguay,Argentina,USA,Yugoslavia,70,13,18,590549,5,Win,1
3,201,1096,MEX,LUQUE Juan (MEX),Starting,Unknown,Juan CARRENO,Unknown,G70',1930,1930-07-13 15:00:00,Group 1,Pocitos,Montevideo,France,4,1,Mexico,Normal,4444,3,0,LOMBARDI Domingo (URU),FRA,MEX,Uruguay,Uruguay,Argentina,USA,Yugoslavia,70,13,18,590549,5,Loss,1
4,201,1096,FRA,CAUDRON Raoul (FRA),Starting,Unknown,Ernest LIBERATI,Unknown,No Event,1930,1930-07-13 15:00:00,Group 1,Pocitos,Montevideo,France,4,1,Mexico,Normal,4444,3,0,LOMBARDI Domingo (URU),FRA,MEX,Uruguay,Uruguay,Argentina,USA,Yugoslavia,70,13,18,590549,5,Win,0


---
## 8. Transformation Log

In [24]:
log = pd.DataFrame({
    'Step': range(1, 15),
    'Column(s)': [
        'All rows', 'All columns', 'assistant_1, assistant_2', 'datetime',
        'home/away goals, half-time goals', 'attendance', 'win_conditions',
        'shirt_number', 'position', 'event', 'line_up',
        'tournament_attendance', 'All string cols',
        'total_goals, match_result, is_goal_scorer'
    ],
    'Action': [
        'Removed 736 duplicate rows',
        'Renamed all columns to snake_case',
        'Dropped (no analytical value)',
        'Parsed string to datetime64 format',
        'Cast float64 to int (blank rows had forced float)',
        'Filled 92 nulls with year-level median; cast to int',
        'Replaced empty string with Normal (regular-time wins)',
        'Replaced placeholder 0 with NaN',
        'Filled 33641 NaN with Unknown (not recorded historically)',
        'Filled 28715 NaN with No Event (no recorded action)',
        'Decoded S to Starting, N to Substitute',
        'Removed European thousands-separator dot; cast to int',
        'Stripped leading/trailing whitespace',
        'Added 3 derived KPI columns'
    ]
})
print(log.to_string(index=False))

 Step                                 Column(s)                                                    Action
    1                                  All rows                                Removed 736 duplicate rows
    2                               All columns                         Renamed all columns to snake_case
    3                  assistant_1, assistant_2                             Dropped (no analytical value)
    4                                  datetime                        Parsed string to datetime64 format
    5          home/away goals, half-time goals         Cast float64 to int (blank rows had forced float)
    6                                attendance       Filled 92 nulls with year-level median; cast to int
    7                            win_conditions     Replaced empty string with Normal (regular-time wins)
    8                              shirt_number                           Replaced placeholder 0 with NaN
    9                                  positio

---
## 9. Export Clean Master Dataset

In [25]:
df.to_csv('../data/processed/world_cup_master.csv', index=False)

print('Exported: ../data/processed/world_cup_master.csv')
print(f'  Rows    : {df.shape[0]:,}')
print(f'  Columns : {df.shape[1]}')
print(f'  Columns : {list(df.columns)}')

Exported: /content/drive/MyDrive/DVA Capstone II/FIFA/world_cup_master2.csv
  Rows    : 36,594
  Columns : 37
  Columns : ['roundid', 'matchid', 'team_initials', 'coach_name', 'line_up', 'shirt_number', 'player_name', 'position', 'event', 'year', 'datetime', 'stage', 'stadium', 'city', 'home_team_name', 'home_team_goals', 'away_team_goals', 'away_team_name', 'win_conditions', 'attendance', 'half_time_home_goals', 'half_time_away_goals', 'referee', 'home_team_initials', 'away_team_initials', 'host_country', 'winner', 'runners_up', 'third', 'fourth', 'goalsscored', 'qualifiedteams', 'matchesplayed', 'tournament_attendance', 'total_goals', 'match_result', 'is_goal_scorer']


---
## 10. Schema Reference — world_cup_master.csv

| Column | Source Dataset | Type | Description |
|--------|---------------|------|-------------|
| round_id | WorldCupPlayers | int | Tournament round identifier |
| match_id | WorldCupPlayers | int | Unique match ID (join key) |
| team_initials | WorldCupPlayers | str | 3-letter team code |
| coach_name | WorldCupPlayers | str | Coach full name and nationality |
| line_up | WorldCupPlayers | str | Starting or Substitute |
| shirt_number | WorldCupPlayers | float | Jersey number (NaN if unknown) |
| player_name | WorldCupPlayers | str | Player full name |
| position | WorldCupPlayers | str | GK / DF / MF / FW / Unknown |
| event | WorldCupPlayers | str | Goal/card/sub notation or No Event |
| year | WorldCupMatches | int | World Cup edition year |
| datetime | WorldCupMatches | datetime | Match date and kick-off time |
| stage | WorldCupMatches | str | Group stage, Quarter-Final, etc. |
| stadium | WorldCupMatches | str | Stadium name |
| city | WorldCupMatches | str | Host city |
| home_team_name | WorldCupMatches | str | Home team full name |
| home_team_goals | WorldCupMatches | int | Full-time home goals |
| away_team_goals | WorldCupMatches | int | Full-time away goals |
| away_team_name | WorldCupMatches | str | Away team full name |
| win_conditions | WorldCupMatches | str | Normal / Extra Time / Penalties |
| attendance | WorldCupMatches | int | Match attendance |
| half_time_home_goals | WorldCupMatches | int | Half-time home goals |
| half_time_away_goals | WorldCupMatches | int | Half-time away goals |
| referee | WorldCupMatches | str | Referee name and nationality |
| home_team_initials | WorldCupMatches | str | Home team 3-letter code |
| away_team_initials | WorldCupMatches | str | Away team 3-letter code |
| host_country | WorldCups | str | Tournament host country |
| winner | WorldCups | str | Tournament winner |
| runners_up | WorldCups | str | Runner-up team |
| third | WorldCups | str | Third place |
| fourth | WorldCups | str | Fourth place |
| goals_scored | WorldCups | int | Total tournament goals |
| qualified_teams | WorldCups | int | Teams that qualified |
| matches_played | WorldCups | int | Total matches in tournament |
| tournament_attendance | WorldCups | int | Total tournament attendance |
| total_goals | Derived | int | home_goals + away_goals |
| match_result | Derived | str | Win / Loss / Draw from player team view |
| is_goal_scorer | Derived | int | 1 if player scored, else 0 |